# QPU Benchmarking Notebook

This notebook benchmarks verified IBM Quantum hardware experiments from the Zenodo dataset (DOI: 10.5281/zenodo.20749395).

**Experiments included:**
- Bell State Entanglement (Heron R2 vs Heron R1)
- BB84 QKD (baseline vs eavesdropping)
- Variational Quantum Circuit (VQC)
- Quantum Random Number Generation (QRNG)
- Reproducibility tests

**Metrics computed:**
- Fidelity comparison
- Noise ratio
- QBER analysis
- Shannon entropy
- Chi-square reproducibility
- Wilson confidence intervals
- QPU time analysis


In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare
from statsmodels.stats.proportion import proportion_confint

sns.set(style='whitegrid', context='notebook', rc={'figure.dpi': 120})
plt.rcParams['figure.figsize'] = (8, 4)

DATA_CSV = '../data/quantum_results_verified.csv'
FIG_DIR = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)

## Load Verified Dataset
If the CSV is missing, synthetic data is used for demonstration.

In [ ]:
if os.path.exists(DATA_CSV):
    df = pd.read_csv(DATA_CSV)
    print(f'Loaded {len(df)} rows from {DATA_CSV}')
else:
    df = pd.DataFrame([
        {'experiment':'Bell','backend':'ibm_fez','shots':1000,'fidelity':0.963,'qber':np.nan,'entropy':np.nan,'qpu_time_s':0.6},
        {'experiment':'Bell','backend':'ibm_torino','shots':1000,'fidelity':0.880,'qber':np.nan,'entropy':np.nan,'qpu_time_s':0.6},
        {'experiment':'BB84','backend':'ibm_fez','shots':1000,'fidelity':np.nan,'qber':0.006,'entropy':np.nan,'qpu_time_s':0.5},
        {'experiment':'BB84_eve','backend':'ibm_fez','shots':1000,'fidelity':np.nan,'qber':0.490,'entropy':np.nan,'qpu_time_s':0.5},
        {'experiment':'VQC','backend':'ibm_fez','shots':1000,'fidelity':np.nan,'qber':np.nan,'entropy':3.885,'qpu_time_s':1.2},
        {'experiment':'QRNG','backend':'ibm_fez','shots':1000,'fidelity':np.nan,'qber':np.nan,'entropy':7.95,'qpu_time_s':0.4}
    ])
    print('Using synthetic dataset.')

df.head()

## Summary Statistics

In [ ]:
summary = df.groupby(['experiment','backend']).agg(
    shots_total=('shots','sum'),
    fidelity_mean=('fidelity','mean'),
    qber_mean=('qber','mean'),
    entropy_mean=('entropy','mean'),
    qpu_time_s=('qpu_time_s','sum')
).reset_index()
summary

## Backend Comparison — Bell Fidelity & Noise Ratio

In [ ]:
bell = df[df['experiment']=='Bell']
if len(bell)>=2:
    r2 = bell[bell['backend']=='ibm_fez']['fidelity'].values[0]
    r1 = bell[bell['backend']=='ibm_torino']['fidelity'].values[0]
    noise_ratio = (1-r1)/(1-r2)
    print('Heron R2 fidelity:', r2)
    print('Heron R1 fidelity:', r1)
    print('Noise ratio (R1/R2):', noise_ratio)
else:
    print('Bell data missing.')

### Plot Fidelity Comparison

In [ ]:
if len(bell)>=2:
    plt.bar(['Heron R2','Heron R1'], [r2, r1], color=['#2c7bb6','#d7191c'])
    plt.ylim(0.7,1.0)
    plt.title('Bell State Fidelity Comparison')
    plt.ylabel('Fidelity')
    plt.show()

## QBER Analysis (BB84)

In [ ]:
bb84 = df[df['experiment'].astype(str).str.contains('BB84')]
bb84

### Plot QBER

In [ ]:
if len(bb84)>0:
    plt.bar(bb84['experiment'], bb84['qber'], color=['green','red'][:len(bb84)])
    plt.title('BB84 QBER Comparison')
    plt.ylabel('QBER')
    plt.xticks(rotation=30)
    plt.show()

## VQC Entropy Analysis

In [ ]:
vqc = df[df['experiment']=='VQC']
if len(vqc)>0:
    entropy = vqc['entropy'].values[0]
    plt.bar(['Max (4 bits)','Measured'], [4.0, entropy], color=['gray','#2c7bb6'])
    plt.title('VQC Shannon Entropy')
    plt.ylabel('Entropy (bits)')
    plt.ylim(3.5,4.1)
    plt.show()

## QRNG Distribution (Synthetic Example)

In [ ]:
samples = np.random.randint(16,255,1000)
plt.hist(samples, bins=32, color='#2c7bb6')
plt.title('QRNG Output Distribution (Synthetic)')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()

## Reproducibility — Chi-square Test

In [ ]:
dist_a = np.array([480,20,20,480])
dist_b = np.array([460,40,40,460])

stat, p = chisquare(dist_a, dist_b)
print('Chi-square:', stat)
print('p-value:', p)

## Wilson Confidence Intervals (Fidelity)

In [ ]:
def wilson_ci(fidelity, shots):
    successes = int(fidelity * shots)
    return proportion_confint(successes, shots, method='wilson')

if len(bell)>=2:
    print('Heron R2 CI:', wilson_ci(r2, 1000))
    print('Heron R1 CI:', wilson_ci(r1, 1000))

## QPU Time Analysis

In [ ]:
time_summary = df.groupby('backend').agg(
    total_qpu_time_s=('qpu_time_s','sum'),
    total_shots=('shots','sum')
).reset_index()

time_summary['avg_time_per_shot_ms'] = (time_summary['total_qpu_time_s'] / time_summary['total_shots']) * 1000
time_summary

## Export Benchmark Summary

In [ ]:
summary.to_csv('benchmark_summary.csv', index=False)
print('Saved benchmark_summary.csv')

### End of Notebook
This notebook provides a complete benchmarking pipeline for verified IBM Quantum hardware experiments.